# Module 06 - Voice
## Voice Recognition and Generation
Voice recognition and generation technologies, have seen remarkable advancements in recent years. Voice recognition involves converting spoken language into text or commands through sophisticated algorithms that process phonemes, accents, and linguistic patterns. This technology powers voice assistants like Alexa and Siri, enabling natural interactions between humans and machines. On the other hand, voice generation refers to creating synthetic speech from text, often mimicking human-like qualities such as intonation and emotion. These systems leverage neural networks like recurrent neural networks (RNNs) or more advanced models, such as transformers, to produce realistic and dynamic speech outputs. Together, these innovations are driving progress in accessibility, communication, and automation.

<p style="text-align: center"><img src="https://thislondonhouse.com/Images/soundwave.png" style="width: 500px;"></p>

In the business world, voice recognition and generation technologies are transforming how organizations operate and interact with customers. For instance, customer service is becoming increasingly automated with AI-powered voice bots that handle inquiries, troubleshoot issues, and offer solutions without human intervention, dramatically improving efficiency and availability. Businesses are also leveraging voice generation for marketing campaigns, creating personalized and engaging ads tailored to individual consumers' needs. Furthermore, industries such as healthcare use voice recognition to transcribe medical records, enabling quicker and more accurate documentation. These technologies are unlocking new levels of productivity, while enhancing the customer experience across a wide range of industries.

### Voice Recognition
Voice recognition, often referred to as automatic speech recognition (ASR), is a technology that enables machines to interpret spoken language and convert it into text or commands. It plays a pivotal role in bridging human-machine communication, enabling seamless interactions through voice-controlled devices, virtual assistants, and accessibility tools. The process involves analyzing audio signals to extract meaningful linguistic information, despite challenges like accents, background noise, and varied speech patterns. Over the years, voice recognition systems have grown increasingly sophisticated, leveraging advancements in machine learning and technological advancement to improve accuracy and responsiveness.

When recognizing voice in sound files, several techniques are commonly employed to extract and interpret speech. The process typically begins with acoustic modeling, which uses sound wave analysis to identify phonemes—the smallest units of sound in a language. Features like Mel-frequency cepstral coefficients (MFCCs) are often extracted to capture the unique characteristics of human speech. Then, language modeling comes into play, predicting word sequences based on context and statistical probabilities. Advanced approaches, like deep neural networks (DNNs) or recurrent neural networks (RNNs), are trained on large datasets to learn complex patterns in speech. These models improve the system's ability to handle diverse voices and scenarios, making voice recognition increasingly versatile and reliable.

### Voice Generation
Voice generation, commonly known as text-to-speech (TTS), is a technology that synthesizes spoken language from text input, enabling machines to communicate in human-like voices. It is widely used in applications ranging from virtual assistants and navigation systems to audiobooks and accessibility tools for individuals with visual impairments or reading difficulties. The goal of voice generation is to produce speech that is natural, expressive, and intelligible, creating seamless and effective human-machine interactions. Over the years, advancements in AI have significantly enhanced the quality and realism of synthetic speech, enabling it to adapt to various languages, accents, and emotional tones.

Several techniques are commonly employed in voice generation to produce high-quality speech. Early systems relied on concatenative synthesis, piecing together pre-recorded audio segments to form words and sentences. However, this approach lacked flexibility and sounded robotic. Modern TTS systems leverage deep learning models, such as WaveNet and Tacotron, which use neural networks to generate speech waveforms directly from text. These models analyze linguistic features, such as phonemes and prosody, to produce smooth and natural-sounding speech. Additionally, neural vocoders enhance the final audio quality by refining the synthesized waveforms. With these advanced techniques, voice generation systems can mimic human intonation and emotion, achieving unparalleled realism and adaptability.

### Feature Extraction
When analyzing an audio file, features like MFCCs, pitch, spectral centroid, HNR, and zero-crossing rate provide valuable insights into its characteristics. MFCCs (Mel-Frequency Cepstral Coefficients) capture the timbral features of sound by mimicking how humans perceive audio on the Mel scale, making them critical for applications like speech and speaker recognition. Pitch refers to the fundamental frequency of the audio, which is key for identifying tonal properties and distinguishing between voices or musical notes. The spectral centroid indicates the "center of mass" of the frequency spectrum and is often associated with the brightness of a sound, with higher values signifying sharper tones. HNR (Harmonic-to-Noise Ratio) measures the proportion of harmonic content to noise in the signal, offering insights into the clarity of voiced sounds, which is useful in tasks like speaker characterization. Finally, the zero-crossing rate measures how frequently the waveform crosses the zero amplitude axis, with higher rates typically indicating noisier or higher-pitched sounds. Together, these features offer a detailed representation of the audio's structure and help differentiate between various types of signals effectively.

In [ ]:
# Libraries
import os
import json
import random
from concurrent.futures import ThreadPoolExecutor, as_completed
from time import time
from dotenv import load_dotenv
from groq import Groq
from IPython.display import Audio
from transformers.utils import get_json_schema
import librosa
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt
from sklearn import metrics
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression, RidgeClassifier, SGDClassifier
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier, NearestCentroid
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from tqdm import tqdm

In [ ]:
# Functions
def plot_descriptives(df):
    numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
    categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
    
    # Plot setup
    num_plots = len(numeric_cols) + len(categorical_cols)
    cols = 3
    rows = int(np.ceil(num_plots / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(12, rows * 4))
    axes = axes.flatten()
    
    # Numeric columns: histograms
    for i, col in enumerate(numeric_cols):
        sns.histplot(df[col], bins=10, ax=axes[i])
        axes[i].set_title(f'Distribution of {col}')
        axes[i].set_xlabel(col)
        axes[i].set_ylabel('Frequency')
    
    # Categorical columns: bar charts
    for j, col in enumerate(categorical_cols, start=len(numeric_cols)):
        sns.countplot(x=df[col], ax=axes[j])
        axes[j].set_title(f'Counts of {col}')
        axes[j].set_xlabel(col)
        axes[j].set_ylabel('Count')
        axes[j].tick_params(axis='x', rotation=45)
    
    # Remove unused axes
    for k in range(num_plots, len(axes)):
        fig.delaxes(axes[k])
    
    plt.tight_layout()
    plt.show()
    
def extract_audio_features(audio_file_data):
    audio_file_label = audio_file_data['label']
    audio_file = audio_file_data['file']
    y, sr = librosa.load(audio_file, sr=None)

    # Spectral Features
    spectral_centroid = librosa.feature.spectral_centroid(y=y, sr=sr)
    spectral_bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr)
    spectral_contrast = librosa.feature.spectral_contrast(y=y, sr=sr)
    spectral_flatness = librosa.feature.spectral_flatness(y=y)
    spectral_rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)

    # Temporal Features
    zero_crossing_rate = librosa.feature.zero_crossing_rate(y)
    rms_energy = librosa.feature.rms(y=y)

    # Pitch
    pitches, magnitudes = librosa.piptrack(y=y, sr=sr)
    pitch_values = []
    for t in range(pitches.shape[1]):
        pitch_frame = pitches[:, t]
        max_pitch = np.max(pitch_frame)  # Get the maximum pitch for this frame
        if max_pitch > 0:  # Only consider non-zero pitch values
            pitch_values.append(max_pitch)

    # MFCCs
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)

    # HNR
    harmonic, percussive = librosa.effects.hpss(y)
    harmonic_rms = librosa.feature.rms(S=np.abs(librosa.stft(harmonic)))
    percussive_rms = librosa.feature.rms(S=np.abs(librosa.stft(percussive)))

    features = {
        "label": audio_file_label,
        "mean_spectral_centroid": np.mean(spectral_centroid),
        "mean_spectral_bandwidth": np.mean(spectral_bandwidth),
        "mean_spectral_contrast": np.mean(spectral_contrast),
        "mean_spectral_flatness": np.mean(spectral_flatness),
        "mean_spectral_rolloff": np.mean(spectral_rolloff),
        "mean_zero_crossing_rate": np.mean(zero_crossing_rate),
        "mean_rms_energy": np.mean(rms_energy),
        "mean_pitch": np.mean(pitch_values) if pitch_values else None,
        "min_pitch": min(pitch_values) if pitch_values else None,
        "max_pitch": max(pitch_values) if pitch_values else None,
        "mean_mfcc": np.mean(np.mean(mfccs, axis=1)),
        "mean_hnr": np.mean(harmonic_rms / (percussive_rms + 1e-6))
    }

    return features

def process_audio_folder(data_folder):
    results = []
    for folder in ['female', 'male']:
        folder_path = os.path.join(data_folder, folder)
        if not os.path.exists(folder_path):
            print(f"Folder {folder_path} does not exist.")
            continue

        audio_files = [{'file': file_path, 'label': folder} for file_path in
                       [os.path.join(folder_path, file) for file in os.listdir(folder_path) if file.endswith('.wav')]]

        with tqdm(total=len(audio_files), desc=f"Processing {folder.upper()} Audio Files") as progress_bar:
            with ThreadPoolExecutor() as executor:
                futures = {executor.submit(extract_audio_features, file): file for file in audio_files[:]}
                for future in as_completed(futures):
                    progress_bar.update(1)
                    results.append(future.result())

    return pd.DataFrame(results)

def visualize_audio_file(file_path):
    y, sr = librosa.load(file_path)

    # Plot the waveform
    plt.figure(figsize=(10, 8))
    plt.subplot(2, 2, 1)
    librosa.display.waveshow(y, sr=sr)
    plt.title('Waveform')
    plt.xlabel('Time (s)')
    plt.ylabel('Amplitude')
    
    plt.subplot(2, 2, 2)
    D = librosa.amplitude_to_db(np.abs(librosa.stft(y)), ref=np.max)
    librosa.display.specshow(D, sr=sr, x_axis='time', y_axis='log')
    plt.colorbar(format='%+2.0f dB')
    plt.title('Spectrogram')
    
    plt.subplot(2, 2, 3)
    mel_spectrogram = librosa.feature.melspectrogram(y=y, sr=sr)
    mel_spectrogram_db = librosa.amplitude_to_db(mel_spectrogram, ref=np.max)
    librosa.display.specshow(mel_spectrogram_db, sr=sr, x_axis='time', y_axis='mel')
    plt.colorbar(format='%+2.0f dB')
    plt.title('Mel Spectrogram')

    plt.subplot(2, 2, 4)
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    librosa.display.specshow(mfccs, sr=sr, x_axis='time')
    plt.colorbar()
    plt.title('MFCC')
    plt.tight_layout()
    plt.show()

def classifier_performance(y, y_pred, labels=None):
    accuracy = metrics.accuracy_score(y, y_pred)
    precision = metrics.precision_score(y, y_pred, average='weighted')
    recall = metrics.recall_score(y, y_pred, average='weighted')
    balanced_accuracy = metrics.balanced_accuracy_score(y, y_pred)
    f1 = metrics.f1_score(y, y_pred, average='weighted')
    report = metrics.classification_report(y, y_pred, target_names=[i for i in sorted(
        labels)] if not labels is None else np.unique(y_pred))

    # Display the confusion matrix with custom labels
    conf_matrix = metrics.confusion_matrix(y, y_pred)
    disp = metrics.ConfusionMatrixDisplay(confusion_matrix=conf_matrix, display_labels=[i for i in sorted(
        labels)] if not labels is None else np.unique(y_pred))
    disp.plot(cmap=plt.cm.Greens)

    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"Balanced Accuracy: {balanced_accuracy:.4f}")
    print(f"F1-score: {f1:.4f}")
    print("\nDetailed Classification Report:")
    print(report)
    plt.show()

def feature_importance(model_pipeline, preprocessor_name, model_name):
    all_feature_names = []
    [all_feature_names.extend(transformer.get_feature_names_out()) for key, transformer in model_pipeline.named_steps[preprocessor_name].named_transformers_.items()]
    pd.DataFrame(
        model_pipeline[model_name].coef_[0] * np.std(model_pipeline[preprocessor_name].fit_transform(X_train), axis=0),
        columns=["Coefficients"],
        index=all_feature_names
    ).plot(kind="barh", figsize=(7, 8))
    plt.title("Feature Importance")
    plt.show()

## Voice Exercise 1
### Business Problem
Being able to discriminate sounds is an essential first step to teaching a computer how to hear. If we are able to effectively teach a computer to discriminate between different types of sounds, we could then build subsequent tools that leverage these ablilities and automate processes that previously required human intervention.

### Data Collection/Selection
We will be loading data from a kaggle dataset. More information here: https://www.kaggle.com/datasets/murtadhanajim/gender-recognition-by-voiceoriginal

The data are organized into folders containing clips of male voices and clips of female voices. As with the images exercise, the folder will serve as the class we seek to predict.

In [ ]:
data_folder = '/pub/Data/male-female'

Let's take a look at a random sample of audio files so we understand what we are analyzing.

In [ ]:
selected_folder = random.choice(["male", "female"])  # Randomly select a folder
selected_file = random.choice([f for f in os.listdir(os.path.join(data_folder, selected_folder)) if f.endswith((".mp3", ".wav", ".ogg"))])
audio_file = os.path.join(data_folder, selected_folder, selected_file)

In [ ]:
display(Audio(audio_file))

We cannot analyze sound files directly so we must first convert the audio files into features that we will use to classify audio samples. These plots visualize the characteristics of the sound file. We will extract these features and use them in our classifers.

In [ ]:
visualize_audio_file(audio_file)

Next, we will process the files in our data folders and extract the relevant features. This can take a long time, so we will try to load a csv. If the csv does not exist, we will process the audio files and create the csv for future analysis.

In [ ]:
try:
    audio_df = pd.read_csv('data/male_female_audio_data.csv')
except FileNotFoundError:
    audio_df = process_audio_folder(data_folder)
    audio_df.to_csv('data/male_female_audio_data.csv', index=False)

With the files loaded, let's take a look at the data we've created.

In [ ]:
audio_df.info(verbose=True, show_counts=True)

As you can see, we have several metrics which describe each audio file. As we are concerned only with classifying the voice in the audio file, we will focus on the overall characteristics of the voice and exclude any temporal characteristics. If we were predicting words, time would be an important characteristic that we would want to include in our analysis.

In [ ]:
audio_df.head()

All of our value are numeric so we will include each column as a numeric column so that we can transform the values appropriately.

In [ ]:
audio_df.describe()

In [ ]:
plot_descriptives(audio_df)

Next, we will collect the features that we wish to analyze and create a dataframe that contains only the features we wish to analyze.

In [ ]:
target_cols = ["label"]
numeric_cols = ["mean_spectral_centroid","mean_spectral_bandwidth","mean_spectral_contrast","mean_spectral_flatness","mean_spectral_rolloff","mean_zero_crossing_rate","mean_rms_energy","mean_pitch","min_pitch","max_pitch","mean_mfcc","mean_hnr"]
categorical_cols = []
count_cols = []
input_cols = [x for x in categorical_cols + numeric_cols + count_cols if x not in target_cols]
data_cols = input_cols + target_cols
df = audio_df[data_cols]

A heatmap of our features will help us understand the relationship between features and identify any highly correlated features. There appears to be some correlation among , but the correlations are not perfect and the data appear to be sufficient for analysis.

In [ ]:
sns.heatmap(df.select_dtypes('number').corr())
plt.show()

This plot illustrates how our classifiers will attempt to segment the data. Orange values represent male voices and blue values represent female voices.

In [ ]:
for target in target_cols:
    sns.pairplot(df, hue=target, corner=True)
    plt.show()

This plot helps us understand the nature of our data. We want to see even distributions with few outliers. This data is well structured for analysis.

In [ ]:
audio_df.hist(figsize=(12, 10), bins=30, edgecolor="black")
plt.show()

There are no records with missing data, but we'll drop missing data out of habit.

In [ ]:
df = df.dropna()

In [ ]:
plot_descriptives(df)

Finally, we will create testing and training samples.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(df[input_cols], df[target_cols], test_size=0.25, random_state=16)

### Model Specification
Unlike the image and text classifiers, our speech classifier relies entirely on numerical values. As such, we will rely on numerical transformers just as classification module. 

We will begin by creating a numeric transformer.

In [ ]:
numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

Then, we add it to our preprocessor

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('cont', numeric_transformer, numeric_cols),
    ])

Next, we build the pipeline. As with previous exercises, we will begin with a logitistic regression classifier.

In [ ]:
# Create a pipeline with the custom transformer
logistic_pipeline = Pipeline([
    ('preprocessor', preprocessor),  # transform numeric data
    ('classifier', LogisticRegression(max_iter=1000))
])

Fit the model

In [ ]:
logistic_pipeline.fit(X_train, np.ravel(y_train))

Predict the testing data

In [ ]:
logistic_predicted = logistic_pipeline.predict(X_test)

Assess model performance.

In [ ]:
classifier_performance(y_test, logistic_predicted)

Identify feature importance.

In [ ]:
feature_importance(logistic_pipeline, 'preprocessor', 'classifier')

As with previous modules, we must then compare our base model against competing classifiers. Here we have a list of classifiers that rely on different techniques for discriminating between records.

In [ ]:
results = []
classifiers = ((DummyClassifier(), "Dummy Classifier"),
               (LogisticRegression(C=5, max_iter=10000), "Logistic Regression"),
               (RidgeClassifier(alpha=1.0, solver="sparse_cg"), "Ridge Classifier"),
               (KNeighborsClassifier(n_neighbors=100), "kNN"),
               (RandomForestClassifier(), "Random Forest"),
               (GaussianNB(), "Gaussian Niave Bayes"),
               (SVC(kernel='linear', C=1.0, max_iter=100000), "Linear SVC"),
               (SGDClassifier(loss="log_loss", alpha=1e-4, n_iter_no_change=3, early_stopping=True), "log-loss SGD",),
               (NearestCentroid(), "NearestCentroid"),
              )

We run through all of the classifiers to assess performance.

In [ ]:
for clf, name in classifiers:
    print("=" * 80)
    print(name)
    print("_" * 80)
    print("Training: ")
    print(clf)
    t0 = time()
    if name == 'Neural Network':
        ## no need to flatten for neural network
        pipeline = Pipeline([
            ('scaler', StandardScaler()),  # Scale features
            ('classifier', clf)
        ])
    else:
        pipeline = Pipeline([
            ('preprocessor', preprocessor),  # transform numeric data
            ('classifier', clf)
        ])
    pipeline.fit(X_train, np.ravel(y_train))

    train_time = time() - t0
    print(f"train time: {train_time:.3}s")

    t0 = time()
    y_pred = pipeline.predict(X_test)
    test_time = time() - t0
    print(f"test time:  {test_time:.3}s")
    classifier_performance(y_test, y_pred, np.unique(y_test))
    print()
    if name:
        clf_descr = str(name)
    else:
        clf_descr = clf.__class__.__name__

    results.append((clf_descr, metrics.accuracy_score(y_test, y_pred), train_time, test_time))

In [ ]:
results = [[x[i] for x in results] for i in range(4)]

clf_names, score, training_time, test_time = results
training_time = np.array(training_time)
test_time = np.array(test_time)

fig, (ax1, ax2) = plt.subplots(nrows=2, ncols=1, figsize=(10, 8))
ax1.scatter(score, training_time, s=60)
ax1.set(
    title="Score-training time trade-off",
    yscale="log",
    xlabel="test accuracy",
    ylabel="training time (s)",
)
ax2.scatter(score, test_time, s=60)
ax2.set(
    title="Score-test time trade-off",
    yscale="log",
    xlabel="test accuracy",
    ylabel="test time (s)",
)

for i, txt in enumerate(clf_names):
    ax1.annotate(txt, (score[i], training_time[i]))
    ax2.annotate(txt, (score[i], test_time[i]))

plt.tight_layout()
plt.show()

In [ ]:
display(Audio('data/welcome.wav'))

In [ ]:
welcome_df = pd.DataFrame(extract_audio_features({'file': 'data/welcome.wav', 'label': ''}), index=[0])[numeric_cols]
logistic_pipeline.predict(welcome_df)

### Conclusion
The model performed very well. All models were capable of discriminating between male and female voices at a greater than 90% accuracy with some models reaching 97% accuracy. It is not obvious that computers would need to discriminate between voices based on gender, but there are circumstances when gender differences have service implications. In medical settings, diagnoses and prescription differ across genders. The same is true for retail and/or marketing environments where people of different genders have markedly differnet shopping preferences. Therefore, it is clear that our model would be of value in such circumstances.

It is also important to note the nature of our data. The data appear to be pulled from high-quality recordings for men and women reading passages from books. These are clearly artificial settings and would diverge from real-life speech in many ways included tone, pace, clarity, and background noise. It is not clear that our model would perform as well in a more complex voice environment.

## Voice Exercise 2
In this exerise, we will be building an LLM-wrapper application. These steps will serve as a model for how we approach LLM-wrappers in the future.  

### Business Problem
Voice assistants are popular technological tools. Increasingly businesses are seeking to pair the function of voice assistants with the flexibility of LLMs. For example, Yum! Brands (the owner of Taco Bell) is [exploring the ability of LLMs to serve in order-taking roles](https://www.tacobell.com/newsroom/yum-brands-to-expand-voice-ai-technology?msockid=12692b6bdda06bde3fda3fe8dccf6a61). Presumably, this could increase throughput, consistency, and customer service. We will explore how LLMs can both listen and speak.

### Data Collection/Selection
For this exercise, all data will be generated during the interaction with the LLM.

In [ ]:
load_dotenv('variables.env')
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

### LLM Engineering
In this exercise, the LLM is our intelligence, but we have to tell it what kind of intelligence to exhibit. To do this, we will be using to LLMs in concert. We will use a standard chat LLM to handle user interactions and we will use a voice recognition/generation LLM to process speech.

To facilitate this exercise, we need three LLM capabilities: 1) the ability to convert audio repsonses into text; 2) the ability to convert text responses into audio; 3) the ability to 'chat.' Before putting all three pieces together, we will look at the first two, independently (see the module on text for more details on chat capabilities).
#### Speech to Text
Speech-to-text models process audio files and seek to provide a transcript of what was said in the file. We will be using OpenAI's Whisper model for this task. First, we set the location of the file we want to transcribe.

In [ ]:
filename = "data/welcome.wav"

Then, we will open the file, read the contents of the file and pass that into a transcription request to the Whisper model.

In [ ]:
with open(filename, "rb") as audio_file:
    transcription = client.audio.transcriptions.create(
      file=audio_file,
      model="whisper-large-v3",
      temperature=0,
      response_format="text",
    )
    print(transcription)

#### Text to Speech
Text-to-speech models perform the reverse task. That is, these models ingest text and try to produce a spoken facsimile. To illustrate, we will reverse engineer the welcome message above. First, we will define the text that we want to be spoken

In [ ]:
welcome_text = "Welcome to IS 452, Artificial Intelligence in Business."

Next, we will use the Orpheus model to conver the text into an audio file. In the example below, we use the 'Troy' voice, but there are several voices available and more information is available here: [voices](https://console.groq.com/docs/text-to-speech/orpheus#available-voices)

In [ ]:
response = client.audio.speech.create(
    model="canopylabs/orpheus-v1-english",
    voice="troy",
    input=welcome_text,
    response_format="wav"
)

The response contains audio data, but we need to write it to a file before we can hear the result.

In [ ]:
response.write_to_file("data/orpheus-welcome.wav")
display(Audio("data/orpheus-welcome.wav"))

### Application Building
We will put these techniques together to build an AI order entry bot. We will begin by defining the identity of the LLM. In this case, we want the LLM to present as a customer service representative so we will embed some basic customer-service knowledge into its prompt.

In [ ]:
system_prompt = {
    "role": "system",
    "content": """
You are a order-taking assistant at the Chez AI bakery. 
Your name is Remi. 
Our menu contains the following products: croissant, brownie, cappuccino.
Do not make up items that are not on our menu.
Do not make up prices.
Your responses should be courteous and succinct.
Before asking to help a customer, introduce yourself, welcome the customer, and ask for their name. 
Always confirm the order.
Do not forget to report how much is owed 
be sure to end the conversation with 'It was my pleasure to help you today'.
    """
}

To simplify the flow of the interaction, we will be leveraging two functions that convert text to speech and speech to text. The first function sends an audio file to groq to request a transcription of what is said. The function produces a transcript of the file.

In [ ]:
def transcribe_audio_file(audio_file_path):
    with open(audio_file_path, "rb") as file:
        transcription = client.audio.transcriptions.create(
          file=file, # Required audio file
          model="whisper-large-v3-turbo", # Required model to use for transcription
          response_format="text",  # Optional
          language="en",  # Optional
          temperature=0.0  # Optional
        )
    return transcription.strip()

The next function takes a text response and generates an audio recording of the response. The function sends the text and the preferred voice to the LLM and requests a recording. The function generates a file and returns the path to the file.

In [ ]:
def generate_spoken_response(text_response, voice):
    speech_file_path = "spoken_response.wav" 

    spoken_response = client.audio.speech.create(
        model="canopylabs/orpheus-v1-english",
        voice=voice,
        input=text_response,
        response_format="wav"
    )

    spoken_response.write_to_file(speech_file_path)
    return speech_file_path

In this context, we do not want the LLM to improvise, allowing the user to order anything they desire. We could try to limit the LLM by embedding a menu in the system prompt, but menus and prices can be rather dynamic and it may be cumbersome to redefine our system prompt everytime a price changes or an item is added to the menu. Instead, we will create an inventory function that looks up products. We will tell the LLM how to use the function and what it does and give it the autonomy to use the function in situations where it lacks sufficient information. It is important to note, the execution of the fuction happens locally and the result is fed back to the LLM, but the decision to call the function is left to the LLM.

#### Tool Use
In 2023, OpenAI introduced [function calling](https://openai.com/index/function-calling-and-other-api-updates/). This feature allows LLMs to call programatic functions if the user has requested capabilities that the LLM lacks. For example, if the user asked an LLM: "What's the weather like in New York?" the LLM would probably respond with a general overview of typical weather in New York. However, if we give the LLM a function that takes a given location and looks up the current weather, the LLM could call that function in the event the user asks for something that the support function is capable of handling. Function calling has evolved into a more agnostic feature known as Tooling. Tools can be created that may aid the LLM and the LLM has autonomy to use a tool when it deems necessary.

This function retrieves product prices. The function requests the name of a product and the desired quantity and returns current pricing for those items in the given quantity. This function will be attached to chat requests in the event the user requests product pricing.

In [ ]:
def get_bakery_prices(bakery_item: str, item_quantity: int):
    """
    Returns the prices for a given bakery product.

    Args:
        bakery_item: The name of the bakery item,
        item_quantity: The quantity of the bakery item
    """
    if "croissant" in bakery_item:
        price = 4.25 * item_quantity
    elif "brownie" in bakery_item:
        price = 2.50 * item_quantity
    elif "cappuccino" in bakery_item:
        price = 4.75 * item_quantity
    else:
        price = "Sold Out"

    return json.dumps({"price": price})

Here, we collect the available tools into a list of tools that will then be passed to the LLM.

In [ ]:
tools = [get_json_schema(get_bakery_prices)]

available_functions = {
    "get_bakery_prices": get_bakery_prices,
}

In [ ]:
chat_model = "meta-llama/llama-4-scout-17b-16e-instruct"
chat_history = [system_prompt]

while True:
    # Make the initial API call to Groq
    chat_response = client.chat.completions.create(
        model=chat_model, # LLM to use
        messages=chat_history, # Conversation history
        stream=False,
        tools=tools, # Available tools (i.e. functions) for our LLM to use
        tool_choice="auto", # Let our LLM decide when to use tools
        max_tokens=4096 # Maximum number of tokens to allow in our response
    )

    if chat_response.choices[0].message.tool_calls:
        chat_history.append(chat_response.choices[0].message)
        # chat_history.append({'role':'assistant', 'tool_calls':[t.model_dump() for t in chat_response.choices[0].message.tool_calls]})
        # Process each tool call
        
        for tool_call in chat_response.choices[0].message.tool_calls:
            function_name = tool_call.function.name
            function_to_call = available_functions[function_name]
            function_args = json.loads(tool_call.function.arguments)
            # Call the tool and get the response
            function_response = function_to_call(
                bakery_item=function_args.get("bakery_item"),
                item_quantity=function_args.get("item_quantity")
            )
            # Add the result of the tool call to the conversation
            chat_history.append(
                {
                    "tool_call_id": tool_call.id, 
                    "role": "tool", # Indicates this message is from tool use
                    "name": function_name,
                    "content": function_response,
                }
            )
            
        # Make a second API call with the updated conversation
        chat_response = client.chat.completions.create(
            model=chat_model,
            messages=chat_history
        )
        # Return the final response
        # chat_response = tool_response.choices[0].message.content
        chat_history.append({
            "role": "assistant",
            "content": chat_response.choices[0].message.content
        })
    else:
        chat_history.append({
            "role": "assistant",
            "content": chat_response.choices[0].message.content
        })

    print(chat_response.choices[0].message.content)
    #display(Audio(generate_spoken_response(chat_response.choices[0].message.content, "troy")))

    if "my pleasure" in chat_response.choices[0].message.content.lower():
        break

    user_input = input("> ")
    # display(Audio(user_input))
    # user_input = transcribe_audio_file(user_input)
    chat_history.append({"role": "user", "content": user_input})
    print(user_input)


In [ ]:
chat_history

In [ ]:
chat_history

In [ ]:
chat_response.choices[0].message